# 09f topic and model-score review: sample 60 original comments

This notebook samples 60 comments from 09f for close reading and exports one Excel worksheet. It uses the full cleaned `clean_text`, not text truncated for clustering. It does not generate AI labels or estimate model accuracy.

For consistency with the earlier analyses, split by `post_id`, then calculate the 90th-percentile cutoffs for `p_support` and `p_skeptic` on training comments only. Sample from held-out posts: check skepticism scores in C2 and C4 and support scores in C5. For each topic, select 10 comments above and 10 below the relevant cutoff: 60 comments from 60 distinct posts.

`not_target_high` only means **below the high-score cutoff for that dimension**. It does not mean a low score or an opposite stance.


In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit

DATA_PATH = Path(r"E:\Dataminingproject\Data\09f_interpretable_assignments_with_two_scores.parquet")
OUTPUT_XLSX = Path(r"E:\Dataminingproject\ARM_results\after_09f\topic_stance_60_original_comments.xlsx")
SEED = 42
TARGETS = [(2, "skeptic"), (4, "skeptic"), (5, "support")]
N_PER_GROUP = 10

columns = ["comment_id", "post_id", "subreddit", "primary_cluster", "topic_name_en",
           "clean_text", "p_support", "p_skeptic"]
df = pd.read_parquet(DATA_PATH, columns=columns)
assert len(df) == 28_230 and df["comment_id"].is_unique
assert df["post_id"].notna().all() and df[["p_support", "p_skeptic"]].notna().all().all()
assert df["clean_text"].notna().all() and df["clean_text"].str.strip().ne("").all()

train_idx, test_idx = next(GroupShuffleSplit(
    n_splits=1, test_size=0.2, random_state=SEED
).split(df, groups=df["post_id"]))
train, test = df.iloc[train_idx], df.iloc[test_idx]
assert set(train["post_id"]).isdisjoint(set(test["post_id"]))
thresholds = {dim: train[f"p_{dim}"].quantile(0.90) for dim in ("support", "skeptic")}

selected, used_posts = [], set()
for cluster, dimension in TARGETS:
    for high in (True, False):
        pool = test.loc[
            test["primary_cluster"].eq(cluster)
            & test[f"p_{dimension}"].ge(thresholds[dimension]).eq(high)
            & ~test["post_id"].isin(used_posts)
        ]
        candidates = pool.sample(frac=1, random_state=SEED + cluster * 10 + int(high))
        candidates = candidates.drop_duplicates("post_id")
        if len(candidates) < N_PER_GROUP:
            raise ValueError(f"C{cluster} / {dimension} / high={high}: fewer than 10 distinct posts")
        picked = candidates.head(N_PER_GROUP).copy()
        picked["target_dimension"] = dimension
        picked["score_group"] = "target_high" if high else "not_target_high"
        selected.append(picked)
        used_posts.update(picked["post_id"])

sample = pd.concat(selected, ignore_index=True)
sample = sample.sample(frac=1, random_state=SEED).reset_index(drop=True)
sample.insert(0, "review_id", [f"R{i:03d}" for i in range(1, len(sample) + 1)])
assert len(sample) == sample["comment_id"].nunique() == sample["post_id"].nunique() == 60
assert set(sample["post_id"]).isdisjoint(set(train["post_id"]))

print(f"Sampled {len(sample)} comments from {sample['post_id'].nunique()} distinct posts")
print(f"Training: {len(train):,} comments; held-out: {len(test):,} comments")
print(f"High-score cutoffs: p_support ≥ {thresholds['support']:.7f};p_skeptic ≥ {thresholds['skeptic']:.7f}")
display(sample.groupby(["primary_cluster", "target_dimension", "score_group"])
       .size().rename("Comments").reset_index())

Sampled 60 comments from 60 distinct posts
Training: 22,501 comments; held-out: 5,729 comments
High-score cutoffs: p_support ≥ 0.9863894;p_skeptic ≥ 0.8070047


,primary_cluster,target_dimension,score_group,Comments
0,2,skeptic,not_target_high,10
1,2,skeptic,target_high,10
2,4,skeptic,not_target_high,10
3,4,skeptic,target_high,10
4,5,support,not_target_high,10
5,5,support,target_high,10


## Export a readable Excel workbook

The workbook has one worksheet, `Original comments 60`. `clean_text` appears near the front with wrapped text; headers can be filtered, and the formula bar shows the full comment. Model-score groups are sampling conditions, not human stance judgments.


In [3]:
import math

from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

export_columns = ["review_id", "clean_text", "primary_cluster", "topic_name_en",
                  "target_dimension", "score_group", "p_support", "p_skeptic",
                  "subreddit", "comment_id", "post_id"]
export = sample[export_columns]
assert export["clean_text"].str.len().max() <= 32_767, "A comment exceeds Excel's cell-length limit"

OUTPUT_XLSX.parent.mkdir(parents=True, exist_ok=True)
temporary = OUTPUT_XLSX.with_name(OUTPUT_XLSX.stem + ".partial.xlsx")
with pd.ExcelWriter(temporary, engine="openpyxl") as writer:
    export.to_excel(writer, sheet_name="Original comments 60", index=False)
    ws = writer.sheets["Original comments 60"]
    ws.freeze_panes = "B2"
    ws.auto_filter.ref = ws.dimensions
    ws.sheet_view.showGridLines = False
    ws.row_dimensions[1].height = 32

    widths = {"review_id": 12, "clean_text": 95, "primary_cluster": 18,
              "topic_name_en": 43, "target_dimension": 21, "score_group": 22,
              "p_support": 15, "p_skeptic": 15, "subreddit": 20,
              "comment_id": 18, "post_id": 18}
    for col_idx, header in enumerate(export.columns, start=1):
        ws.column_dimensions[get_column_letter(col_idx)].width = widths[header]
        head = ws.cell(1, col_idx)
        head.fill = PatternFill("solid", fgColor="17324D")
        head.font = Font(name="Aptos", size=11, color="FFFFFF", bold=True)
        head.alignment = Alignment(vertical="center", wrap_text=True)

    for row_idx in range(2, ws.max_row + 1):
        text = str(ws.cell(row_idx, 2).value or "")
        ws.row_dimensions[row_idx].height = min(300, max(76, math.ceil(len(text) / 95) * 17))
        for cell in ws[row_idx]:
            cell.fill = PatternFill("solid", fgColor="F1F5F7" if row_idx % 2 == 0 else "FFFFFF")
            cell.font = Font(name="Aptos", size=11, color="17324D")
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            if isinstance(cell.value, str):
                cell.data_type = "s"  # Reddit text must never become an Excel formula.
        for score_col in (7, 8):
            ws.cell(row_idx, score_col).number_format = "0.0000"
        group_cell = ws.cell(row_idx, 6)
        group_cell.fill = PatternFill(
            "solid", fgColor="DDF3E7" if group_cell.value == "target_high" else "FFF0DA"
        )

temporary.replace(OUTPUT_XLSX)
print(f"Saved {len(export)} original comments: {OUTPUT_XLSX}")

Saved 60 original comments: E:\Dataminingproject\ARM_results\after_09f\topic_stance_60_original_comments.xlsx
